# 모듈 07: 임베딩과 벡터 저장소

## 이 모듈에서 배울 것

1. **임베딩(Embedding)**: 텍스트를 숫자 벡터로 변환하는 방법
2. **코사인 유사도**: 두 텍스트가 얼마나 비슷한지 수치로 측정
3. **Chroma 벡터 저장소**: 문서와 벡터를 함께 저장하고 검색
4. **Retriever**: 벡터 저장소를 LCEL 체인에 연결하는 인터페이스
5. **영구 저장**: `persist_directory`로 디스크에 저장하고 재사용

---

## RAG 파이프라인에서의 위치

```
[1단계: 문서 로드] → [2단계: 텍스트 분할] → [3단계: 임베딩] → [4단계: 벡터 저장] → [5단계: 검색] → [6단계: 생성]
      모듈 06                 모듈 06             ★ 모듈 07 ★         ★ 모듈 07 ★         모듈 07          모듈 08
```

이 모듈은 **3단계(임베딩)**와 **4단계(벡터 저장)**를 담당합니다.

## 학습 목표

이 모듈을 완료하면 다음을 할 수 있습니다:

1. `GoogleGenerativeAIEmbeddings`로 텍스트를 768차원 벡터로 변환한다.
2. `Chroma.from_documents()`로 Document 리스트를 벡터 저장소에 저장하고, `similarity_search()`로 의미 기반 검색을 수행한다.
3. `as_retriever()`로 Retriever를 생성하여 LCEL 체인에 연결할 준비를 한다.

## 전체 흐름 — RAG 파이프라인 6단계

```
+------------------+     +------------------+     +------------------+
|  1단계: 로드      |     |  2단계: 분할      |     |  3단계: 임베딩    |
|  TextLoader      | --> | RecursiveChar... | --> | Embeddings       |
|  PDF, Web, ...   |     | chunk_size=500   |     | 텍스트 → 벡터    |
+------------------+     +------------------+     +--------+---------+
                                                            |
+------------------+     +------------------+     +--------v---------+
|  6단계: 생성      |     |  5단계: 검색      |     |  4단계: 저장      |
|  LLM + 프롬프트  | <-- |  Retriever       | <-- |  Chroma          |
|  최종 답변 생성   |     |  관련 청크 반환   |     |  벡터 DB 저장     |
+------------------+     +------------------+     +------------------+
```

**이 모듈**: 3단계(임베딩)와 4단계(저장), 그리고 5단계(검색)의 일부를 다룹니다.

## 임베딩이란?

**비유: 지도 좌표**

지도에서 서울과 인천은 가깝고, 서울과 부산은 멉니다.  
임베딩은 **텍스트의 의미**를 지도 좌표처럼 숫자 벡터로 표현합니다.

```
"고양이"    → [0.12, -0.45, 0.78, ...]  (768개 숫자)
"강아지"    → [0.11, -0.43, 0.81, ...]  (비슷한 좌표 → 의미가 유사)
"자동차"    → [-0.55, 0.22, -0.33, ...] (다른 좌표 → 의미가 다름)
```

**핵심**: 의미가 비슷한 텍스트 = 벡터 공간에서 가까운 좌표

| 구분 | 설명 |
|------|------|
| 입력 | 텍스트 문자열 |
| 출력 | 실수 숫자 배열 (예: 768차원) |
| 의미 | 숫자 배열이 텍스트의 "의미"를 담음 |
| 활용 | 의미 기반 검색, 분류, 클러스터링 |

In [1]:
import os
import math
from dotenv import load_dotenv
from langchain_core.documents import Document

# .env 로드 (07_embeddings/ 기준 상위 디렉토리)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

# API 키 확인
api_key = os.getenv('GOOGLE_API_KEY')
if api_key and api_key != 'your_api_key_here':
    print(f'[OK] GOOGLE_API_KEY 로드 성공: {api_key[:10]}...')
else:
    print('[FAIL] API 키가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.')

# 패키지 임포트 확인
try:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    print('[OK] GoogleGenerativeAIEmbeddings 임포트 성공')
except ImportError:
    print('[FAIL] uv add langchain-google-genai')

try:
    from langchain_chroma import Chroma
    print('[OK] Chroma 임포트 성공')
except ImportError:
    print('[FAIL] uv add langchain-chroma chromadb')

[OK] GOOGLE_API_KEY 로드 성공: AIzaSyAlrJ...
[OK] GoogleGenerativeAIEmbeddings 임포트 성공
[OK] Chroma 임포트 성공


In [2]:
# embed_query(): 단일 텍스트 임베딩
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

text = 'LangChain은 LLM 애플리케이션 개발 프레임워크입니다.'
vector = embeddings.embed_query(text)

print(f'입력 텍스트: {text}')
print(f'벡터 차원: {len(vector)}차원')
print(f'앞 5개 값: {[round(v, 4) for v in vector[:5]]}')
print(f'뒤 5개 값: {[round(v, 4) for v in vector[-5:]]}')
print('[OK] embed_query(): 텍스트 → 벡터 변환 성공')

입력 텍스트: LangChain은 LLM 애플리케이션 개발 프레임워크입니다.
벡터 차원: 3072차원
앞 5개 값: [-0.0151, 0.0185, 0.0256, -0.0731, -0.0432]
뒤 5개 값: [0.0045, -0.0133, 0.0058, -0.0043, 0.0274]
[OK] embed_query(): 텍스트 → 벡터 변환 성공


In [3]:
# embed_documents(): 여러 텍스트 임베딩
texts = [
    'LangChain은 LLM 애플리케이션 개발 프레임워크입니다.',
    'LangChain은 AI 개발에 사용되는 오픈소스 도구입니다.',  # 유사
    '오늘 점심으로 김치찌개를 먹었습니다.',                  # 다름
]
vectors = embeddings.embed_documents(texts)

print(f'임베딩한 텍스트 수: {len(vectors)}개')
print(f'각 벡터 차원: {len(vectors[0])}차원')
print()
for i, (text, vec) in enumerate(zip(texts, vectors), 1):
    preview = text[:30] + '...' if len(text) > 30 else text
    print(f'텍스트 {i}: {preview}')
    print(f'  앞 3차원: {[round(v, 4) for v in vec[:3]]}')
print('[OK] embed_documents(): 복수 텍스트 임베딩 성공')

임베딩한 텍스트 수: 3개
각 벡터 차원: 3072차원

텍스트 1: LangChain은 LLM 애플리케이션 개발 프레임워크...
  앞 3차원: [-0.0222, 0.0193, 0.0062]
텍스트 2: LangChain은 AI 개발에 사용되는 오픈소스 도구...
  앞 3차원: [-0.0214, 0.0157, -0.003]
텍스트 3: 오늘 점심으로 김치찌개를 먹었습니다.
  앞 3차원: [-0.0099, 0.022, 0.0185]
[OK] embed_documents(): 복수 텍스트 임베딩 성공


## 코사인 유사도란?

**비유: 두 벡터 사이의 각도**

두 화살표(벡터)가 같은 방향이면 각도가 0도 → 코사인값 = **1**  
두 화살표가 직각이면 각도가 90도 → 코사인값 = **0**  
두 화살표가 반대 방향이면 각도가 180도 → 코사인값 = **-1**

```
코사인 유사도 = (v1 · v2) / (|v1| × |v2|)

  1.0  →  완전히 동일한 의미
  0.8+ →  매우 유사
  0.5  →  관련 있음
  0.0  →  무관한 의미
 -1.0  →  반대 의미
```

**주의**: Chroma의 `similarity_search_with_score()`는 기본적으로 **L2(유클리드) 거리**를 반환합니다.  
L2 거리는 **낮을수록** 더 유사한 문서입니다.

In [4]:
# 코사인 유사도 직접 계산
def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a * a for a in v1))
    norm2 = math.sqrt(sum(b * b for b in v2))
    return dot / (norm1 * norm2)

sim_12 = cosine_similarity(vectors[0], vectors[1])  # 유사한 주제
sim_13 = cosine_similarity(vectors[0], vectors[2])  # 다른 주제

print(f'텍스트 1: {texts[0][:30]}...')
print(f'텍스트 2: {texts[1][:30]}...')
print(f'텍스트 3: {texts[2]}')
print()
print(f'유사도 (1 vs 2, 비슷한 주제): {sim_12:.4f}')
print(f'유사도 (1 vs 3, 다른 주제  ): {sim_13:.4f}')
if sim_12 > sim_13:
    print('[OK] 코사인 유사도: 유사한 텍스트일수록 1에 가까움')
else:
    print('[확인] 예상과 다른 결과입니다.')

텍스트 1: LangChain은 LLM 애플리케이션 개발 프레임워크...
텍스트 2: LangChain은 AI 개발에 사용되는 오픈소스 도구...
텍스트 3: 오늘 점심으로 김치찌개를 먹었습니다.

유사도 (1 vs 2, 비슷한 주제): 0.9368
유사도 (1 vs 3, 다른 주제  ): 0.7133
[OK] 코사인 유사도: 유사한 텍스트일수록 1에 가까움


## 벡터 저장소란?

**비유: 도서관 색인 카드**

도서관에는 책마다 색인 카드가 있습니다.  
벡터 저장소는 그 색인 카드 시스템과 같습니다.

```
도서관 색인 시스템        벡터 저장소
----------------------    --------------------------
책 (내용)           →     Document (page_content)
색인 카드 (키워드)   →     벡터 (숫자 배열)
책 정보 (저자, 연도) →     metadata (dict)
사서 (검색 담당)     →     Retriever
```

**검색 과정**:
1. 질문을 임베딩 → 질문 벡터 생성
2. 저장된 모든 벡터와 거리 계산
3. 거리가 가까운 Document 반환

키워드 검색과 달리, **의미 기반 검색**이 가능합니다.  
("AI 신경망" 검색 → "딥러닝은 뇌를 모방한다" 문서 찾기)

In [5]:
# Chroma.from_documents(): 벡터 저장소 생성
sample_docs = [
    Document(page_content='파이썬은 배우기 쉬운 프로그래밍 언어입니다.', metadata={'topic': 'python'}),
    Document(page_content='자바스크립트는 웹 개발의 핵심 언어입니다.', metadata={'topic': 'javascript'}),
    Document(page_content='머신러닝은 데이터로부터 패턴을 학습합니다.', metadata={'topic': 'ml'}),
    Document(page_content='딥러닝은 신경망을 사용한 머신러닝 기법입니다.', metadata={'topic': 'dl'}),
    Document(page_content='LangChain은 LLM 애플리케이션 개발 프레임워크입니다.', metadata={'topic': 'langchain'}),
    Document(page_content='RAG는 검색과 생성을 결합한 AI 기법입니다.', metadata={'topic': 'rag'}),
]

vectorstore = Chroma.from_documents(
    documents=sample_docs,
    embedding=embeddings,
)
count = vectorstore._collection.count()
print(f'저장된 Document 수: {count}개')
print(f'저장 내용: page_content + metadata + 벡터가 함께 저장됨')
print('[OK] Chroma.from_documents() 벡터 저장소 생성 성공')

저장된 Document 수: 6개
저장 내용: page_content + metadata + 벡터가 함께 저장됨
[OK] Chroma.from_documents() 벡터 저장소 생성 성공


In [6]:
# similarity_search(): 기본 유사도 검색
query = 'AI와 신경망에 대해 알고 싶어요'
results = vectorstore.similarity_search(query, k=3)

print(f'검색 질문: "{query}"')
print(f'검색 결과 (상위 {len(results)}개):')
print('-' * 55)
for i, doc in enumerate(results, 1):
    print(f'{i}. [{doc.metadata["topic"]:12s}] {doc.page_content}')
print('[OK] similarity_search() 유사도 검색 성공')

검색 질문: "AI와 신경망에 대해 알고 싶어요"
검색 결과 (상위 3개):
-------------------------------------------------------
1. [dl          ] 딥러닝은 신경망을 사용한 머신러닝 기법입니다.
2. [rag         ] RAG는 검색과 생성을 결합한 AI 기법입니다.
3. [ml          ] 머신러닝은 데이터로부터 패턴을 학습합니다.
[OK] similarity_search() 유사도 검색 성공


In [7]:
# similarity_search_with_score(): 점수(거리) 포함 검색
results_with_score = vectorstore.similarity_search_with_score(query, k=3)

print(f'검색 질문: "{query}"')
print(f'점수 포함 검색 결과:')
print('-' * 65)
for doc, score in results_with_score:
    print(f'점수: {score:.4f} | [{doc.metadata["topic"]:12s}] {doc.page_content}')
print()
print('Chroma 기본 거리: L2(유클리드) — 점수가 낮을수록 더 유사함')
print('[OK] similarity_search_with_score() 점수 포함 검색 성공')

검색 질문: "AI와 신경망에 대해 알고 싶어요"
점수 포함 검색 결과:
-----------------------------------------------------------------
점수: 0.5448 | [dl          ] 딥러닝은 신경망을 사용한 머신러닝 기법입니다.
점수: 0.6451 | [rag         ] RAG는 검색과 생성을 결합한 AI 기법입니다.
점수: 0.6818 | [ml          ] 머신러닝은 데이터로부터 패턴을 학습합니다.

Chroma 기본 거리: L2(유클리드) — 점수가 낮을수록 더 유사함
[OK] similarity_search_with_score() 점수 포함 검색 성공


## Retriever란?

**비유: 도서관 사서**

직접 서가를 뒤지는 것(similarity_search)과  
사서에게 부탁하는 것(Retriever)은 결과가 같지만 방식이 다릅니다.

```
similarity_search  →  직접 서가를 뒤짐 (독립 실행)
Retriever.invoke   →  사서에게 부탁 (체인에 연결 가능)
```

**왜 Retriever가 필요한가?**

LangChain의 LCEL 체인은 `invoke()` 메서드를 가진 객체끼리 연결됩니다.  
Retriever는 `invoke()`를 지원하므로 체인에 바로 연결할 수 있습니다.

```python
# 모듈 08에서 배울 내용 (미리보기)
chain = retriever | prompt | llm | output_parser
chain.invoke('질문')  # Retriever → LLM → 답변
```

In [8]:
# as_retriever() + invoke()
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
retrieved = retriever.invoke('프로그래밍 언어를 배우고 싶어요')

print(f'검색 질문: "프로그래밍 언어를 배우고 싶어요"')
print(f'검색된 Document 수: {len(retrieved)}개')
print()
for i, doc in enumerate(retrieved, 1):
    print(f'{i}. [{doc.metadata["topic"]}] {doc.page_content}')
print()
print('similarity_search vs Retriever:')
print('  similarity_search : vectorstore.similarity_search(query, k=3)  — 직접 호출')
print('  Retriever.invoke  : retriever.invoke(query)  — LCEL 체인에 연결 가능')
print('[OK] as_retriever() + invoke() 성공')

검색 질문: "프로그래밍 언어를 배우고 싶어요"
검색된 Document 수: 2개

1. [python] 파이썬은 배우기 쉬운 프로그래밍 언어입니다.
2. [javascript] 자바스크립트는 웹 개발의 핵심 언어입니다.

similarity_search vs Retriever:
  similarity_search : vectorstore.similarity_search(query, k=3)  — 직접 호출
  Retriever.invoke  : retriever.invoke(query)  — LCEL 체인에 연결 가능
[OK] as_retriever() + invoke() 성공


## persist_directory — 디스크 저장

**문제**: `Chroma.from_documents()`로 만든 벡터 저장소는 메모리에만 존재합니다.  
프로그램을 종료하면 사라집니다.

**해결**: `persist_directory`를 지정하면 디스크에 저장됩니다.

```python
# 저장
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory='./chroma_db'  # 이 폴더에 저장
)

# 나중에 로드 (임베딩 재계산 없이)
vectorstore = Chroma(
    persist_directory='./chroma_db',
    embedding_function=embeddings
)
```

**장점**:
- 문서가 많을 때 임베딩 API 비용 절약 (재계산 불필요)
- 서버 재시작 후에도 즉시 검색 가능
- 여러 프로그램에서 같은 저장소 공유 가능

In [9]:
# persist_directory: 저장 → 로드 실습
persist_dir = os.path.join(notebook_dir, 'chroma_db')

# 저장
vectorstore_persist = Chroma.from_documents(
    documents=sample_docs,
    embedding=embeddings,
    persist_directory=persist_dir,
)
count_saved = vectorstore_persist._collection.count()
print(f'저장 완료: {persist_dir}')
print(f'저장된 Document 수: {count_saved}개')

# 새 객체로 로드 (프로그램 재시작 시뮬레이션)
vectorstore_loaded = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings,
)
count_loaded = vectorstore_loaded._collection.count()
print(f'로드 완료: {count_loaded}개 Document 복원')

results_loaded = vectorstore_loaded.similarity_search('머신러닝 관련 내용', k=2)
print(f'로드된 저장소 검색 결과: {len(results_loaded)}개')

if count_loaded == count_saved:
    print('[OK] persist_directory: 저장 → 로드 후 동일 데이터 확인')
else:
    print('[FAIL] 데이터 수 불일치')

저장 완료: /Users/sonny/Desktop/Dev/10X 생산성 에이전트 군단 - 사내 챗봇부터 AI 숏츠 생성기까지 외주 수익화 올인원 패키지/LangChain/07_embeddings/chroma_db
저장된 Document 수: 6개
로드 완료: 6개 Document 복원
로드된 저장소 검색 결과: 2개
[OK] persist_directory: 저장 → 로드 후 동일 데이터 확인


In [10]:
# 전체 파이프라인: 로드 → 분할 → 임베딩·저장 → 검색
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print('=== 전체 파이프라인: 로드 → 분할 → 임베딩·저장 → 검색 ===')

# 1. 로드 (모듈 06 data 재사용, fallback 포함)
txt_path = os.path.normpath(os.path.join(notebook_dir, '..', '06_document_loaders', 'data', 'langchain_intro.txt'))
if os.path.exists(txt_path):
    loader = TextLoader(txt_path, encoding='utf-8')
    docs = loader.load()
    print(f'1단계 [로드] : {len(docs)}개 Document (모듈 06 data 사용)')
else:
    docs = [Document(
        page_content='LangChain은 LLM 애플리케이션 개발 프레임워크입니다. RAG는 검색 증강 생성 기법입니다. 임베딩은 텍스트를 숫자 벡터로 변환합니다.',
        metadata={'source': 'fallback'}
    )]
    print(f'1단계 [로드] : {len(docs)}개 Document (fallback 텍스트 사용)')

# 2. 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)
print(f'2단계 [분할] : {len(chunks)}개 청크')

# 3. 임베딩 + 저장
pipeline_dir = os.path.join(notebook_dir, 'pipeline_db')
vs_pipeline = Chroma.from_documents(chunks, embeddings, persist_directory=pipeline_dir)
print(f'3단계 [임베딩·저장]: {vs_pipeline._collection.count()}개 벡터 저장 완료')

# 4. 검색
pipeline_query = 'LangChain의 주요 기능은 무엇인가요?'
pipeline_results = vs_pipeline.similarity_search(pipeline_query, k=2)
print(f'4단계 [검색] : "{pipeline_query}"')
for i, doc in enumerate(pipeline_results, 1):
    src = os.path.basename(doc.metadata.get('source', 'N/A'))
    print(f'  결과 {i}: ({src}) {doc.page_content[:60]}...')

print()
print('[OK] 전체 파이프라인 실행 완료')
print('다음 단계: Retriever를 LLM과 연결 → RAG 완성 (모듈 08)')

=== 전체 파이프라인: 로드 → 분할 → 임베딩·저장 → 검색 ===
1단계 [로드] : 1개 Document (모듈 06 data 사용)
2단계 [분할] : 4개 청크
3단계 [임베딩·저장]: 4개 벡터 저장 완료
4단계 [검색] : "LangChain의 주요 기능은 무엇인가요?"
  결과 1: (langchain_intro.txt) 주요 기능

LangChain의 핵심 기능은 다음과 같습니다. 첫째, 프롬프트 관리 기능을 통해 재사용 가능...
  결과 2: (langchain_intro.txt) 활용 사례

LangChain은 챗봇, 문서 요약, 코드 생성, 데이터 분석 등 다양한 분야에서 활용됩니다....

[OK] 전체 파이프라인 실행 완료
다음 단계: Retriever를 LLM과 연결 → RAG 완성 (모듈 08)


## 배운 것 정리

### 핵심 개념

| 개념 | 설명 | 비유 |
|------|------|------|
| 임베딩 | 텍스트 → 숫자 벡터 변환 | 지도 좌표 |
| 코사인 유사도 | 두 벡터 사이의 각도 (1=동일, 0=무관) | 화살표 방향 |
| 벡터 저장소 | Document + 벡터 + 메타데이터 저장 | 도서관 색인 |
| Retriever | 벡터 저장소를 LCEL에 연결 | 사서 |
| persist_directory | 디스크에 영구 저장 | 책을 서가에 꽂기 |

### 핵심 코드 패턴

```python
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 1. 임베딩 모델 초기화
embeddings = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')

# 2. 단일 텍스트 임베딩
vector = embeddings.embed_query('텍스트')  # → List[float]

# 3. 복수 텍스트 임베딩
vectors = embeddings.embed_documents(['텍스트1', '텍스트2'])  # → List[List[float]]

# 4. 벡터 저장소 생성
vectorstore = Chroma.from_documents(
    documents=docs,           # List[Document]
    embedding=embeddings,     # 임베딩 모델
    persist_directory='./db'  # 디스크 저장 (선택)
)

# 5. 유사도 검색
results = vectorstore.similarity_search(query, k=3)               # → List[Document]
results = vectorstore.similarity_search_with_score(query, k=3)    # → List[(Document, float)]

# 6. Retriever 생성 (LCEL 연결용)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
docs = retriever.invoke('질문')  # → List[Document]

# 7. 저장된 DB 로드
vectorstore = Chroma(
    persist_directory='./db',
    embedding_function=embeddings
)
```

## 다음 모듈 예고: 모듈 08 — RAG (검색 증강 생성)

드디어 전체 RAG 파이프라인을 완성합니다!

```
모듈 07까지 완성된 것:
  문서 로드 → 텍스트 분할 → 임베딩 → 벡터 저장 → 검색

모듈 08에서 추가할 것:
  검색된 문서 → 프롬프트 조립 → LLM 호출 → 최종 답변
```

### 모듈 08 준비 체크리스트

- [ ] 모듈 07 전체 셀 오류 없이 실행 완료
- [ ] `07_embeddings/chroma_db/` 폴더가 생성되었는가?
- [ ] `07_embeddings/pipeline_db/` 폴더가 생성되었는가?
- [ ] `retriever.invoke()`가 `[OK]`로 출력되었는가?
- [ ] GOOGLE_API_KEY가 `.env`에 정상 설정되어 있는가?

### 모듈 08 핵심 코드 미리보기

```python
# 모듈 08: RAG 체인
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

retriever = vectorstore.as_retriever()
prompt = ChatPromptTemplate.from_template("""
다음 컨텍스트를 참고하여 질문에 답하세요:
{context}

질문: {question}
""")
llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash')

# LCEL 체인 조립
chain = retriever | prompt | llm
answer = chain.invoke('LangChain이란 무엇인가요?')
```

---

**수고하셨습니다!** 모듈 07 완료.